# Advanced packages - LAK (Lake)

Part of the **synthetic-valley advanced-packages** series, which upgrades a calibrated base model's simple boundary conditions to MODFLOW 6 *advanced packages* one notebook at a time. Each notebook loads the shared advanced model from `models/` (creating it from the base model on first use with `mf6_notebook_helpers.load_or_create_advanced_model`) and adds one package, so they can be run independently and in any order - except where a package depends on others.

- **UZF** (`mf6-adv-uzf`) - recharge (RCH) -> Unsaturated Zone Flow
- **MAW** (`mf6-adv-maw`) - wells (WEL) -> Multi-Aquifer Well
- **LAK** (`mf6-adv-lak`) - high-K lake -> Lake package *(this notebook)*
- **SFR** (`mf6-adv-sfr`) - river (RIV) -> Streamflow Routing
- **MVR** (`mf6-adv-mvr`) - Water Mover: runoff and seepage to the lake and stream (requires UZF, LAK, SFR)
- **Processing output** (`mf6-adv-processing`) - run the model and plot results

Replace the high-conductivity cells that approximate the lake with the Lake (LAK) package, which simulates the lake level, how far the lake spreads, and its exchange with the groundwater system.

## What this notebook covers

Build the Lake (**LAK**) package on the shared synthetic-valley model, where it
simulates lake stage and its exchange with the groundwater system.

By the end of this notebook you will be able to:

- say what a lake package represents that high-conductivity cells cannot,
- assemble the LAK input from the data already in the model,
- add the package to the shared model in `models/` and run it, and
- leave the updated model in `models/` for the other notebooks listed above.


## Imports and setup

Import FloPy and the helpers, then choose the temporal resolution
(`sample_frequency`) and the model name.

In [ ]:
import flopy
import numpy as np
from mf6_notebook_helpers import (
    drop_packages,
    load_land_surface,
    load_or_create_advanced_model,
    load_temporal_data,
)

In [ ]:
# exercise: choose the temporal resolution and set the model name
# your code here


## Load or create the advanced model

Load the shared advanced model from `models/` (or create it from the base model on first use), then read the rainfall and evaporation series and the lake: which cells it covers, the elevation of its bed, and its volume and area at each lake level.

In [ ]:
# exercise: load (or create) the shared advanced model and read its stress periods
# your code here


In [ ]:
# exercise: read the rainfall and evaporation series and the lake data
# your code here


## Put the lake on top of the aquifer

In the base model the lake is 153 layer 1 cells with a very high hydraulic conductivity, whose top is the lake surface at its highest, 6.5 m (21.3 ft), and which store water as open water does (a specific yield of 1). The Lake package (**LAK**) keeps the lake above the aquifer instead, so the cells under it go back to being aquifer: their top is the lake bed, and they take the aquifer's hydraulic conductivity and storage. Where the lake bed is at the bottom of layer 1, the layer 1 cell has no thickness and passes water straight down to layer 2 (an `idomain` of -1); the lake leaks to layer 2 there.

In [ ]:
# exercise: make the cells under the lake part of the aquifer, with the lake bed as their top
# your code here

gwf.dis.top = top
gwf.dis.idomain = idomain
gwf.npf.k = k
gwf.npf.k33 = k33
gwf.npf.icelltype = icelltype
gwf.sto.sy = sy
gwf.sto.ss = ss

## Build the LAK package

The lake leaks through its bed to the cell beneath each part of it. The bed leakance, 0.01 per day, is the hydraulic conductivity of the bed divided by its thickness. A table gives the lake volume and surface area at each lake level, so the lake spreads over more of the valley as it rises; it has no outlet. Rain falls on the lake and water evaporates from it each stress period.

In [ ]:
# one vertical connection from the lake to the cell under each part of it
# <ifno> <iconn> <cellid> <claktype> <bedleak> <belev> <telev> <connlen> <connwidth>
bedleak = 0.01  # 1/d
# exercise: build the lake connections, then the LAK package data, one row per lake
# your code here


In [ ]:
# exercise: ask for lake stage and the lake-aquifer exchange as observations
# your code here


In [ ]:
# lake stress period data with rainfall and evaporation
# exercise: build the LAK stress period data from the rainfall and evaporation series
# your code here


In [ ]:
# the lake volume and surface area at each lake level
table = list(
    zip(surface["stage"].values, surface["volume"].values, surface["area"].values)
)

# the lake replaces the evaporation of the EVT package and the rain the
# recharge package put on the lake cells (idempotent: drop any prior LAK first)
drop_packages(gwf, "LAK-1", "EVT_0")
for package in list(gwf.packagelist):
    if package.package_type == "laktab":
        gwf.remove_package(package)
rch = gwf.get_package("RCH_0")
if rch is not None:
    rch.stress_period_data = {
        kper: d[d["boundname"] == "land"]
        for kper, d in rch.stress_period_data.get_data().items()
    }
# exercise: create the LAK package and its table of lake volume and area
# your code here

lak.obs.initialize(filename=f"{name}.lak.obs", print_input=True, continuous=lake_obs)

In [ ]:
# rebuild the GWF observation package without the (now redundant) lake head obs
pak = gwf.get_package("GWF-OBS")
if pak is not None:
    gwf_obs = {}
    for key, value in pak.continuous.get_active_key_list():
        if key != f"{name}.lake.obs.csv":
            gwf_obs[key] = value.get_data().tolist()
    drop_packages(gwf, "GWF-OBS")
    flopy.mf6.ModflowUtlobs(
        gwf,
        print_input=True,
        continuous=gwf_obs,
    )
gwf.get_package_list()

## Write and run

Write the updated advanced model back to `models/` and run it to confirm the LAK package is valid.

In [ ]:
# exercise: write the input files and run the model
# your code here

print("LAK advanced model ran successfully")

## Recap

In this notebook you replaced the high-conductivity lake cells and the Evapotranspiration (**EVT**) package with the Lake (**LAK**) package: the cells under the lake became aquifer again, with the lake bed as their top, and the lake now leaks through its bed, rises and spreads according to its table of volume and area, and gains rain and loses evaporation. You wrote the updated advanced model back to `models/` and ran it.